
# ESPAC Livestock LCI -> XML Generator

Generate one XML file per row from the livestock LCI pipeline using a livestock XML template whose exchange `generalComment` values match livestock table columns.

This notebook is independent from the crop XML notebook and inherits the livestock summary strategy from notebook 2 metadata when available.

Important note:
- a dedicated livestock XML template is required at `inputs/04-05_Model_livestock.XML` unless you change the path below;
- this notebook intentionally does not reuse the crop XML model as a silent fallback.


In [7]:

from pathlib import Path
import re
import json
import xml.etree.ElementTree as ET
from datetime import date

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent

DEFAULT_SUMMARY_STRATEGY = 'national'
LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_livestock_filtered_export_summary.json'
summary_meta = {}
if LATEST_FILTERED_SUMMARY_META_PATH.exists():
    try:
        summary_meta = json.loads(LATEST_FILTERED_SUMMARY_META_PATH.read_text(encoding='utf-8')) or {}
    except Exception:
        summary_meta = {}

available_summary_filtered = sorted(
    (PROJECT_DIR / 'outputs/CSVs').glob('02_espac_livestock_lci_table_filtered__summary_*.csv'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
available_summary_filtered_main = [
    p for p in available_summary_filtered
    if ('_uncertainty' not in p.stem) and ('__run_' not in p.stem)
]

_meta_summary = str(summary_meta.get('summary_token', '')).strip() or str(summary_meta.get('summary_level', '')).strip()
meta_filtered_csv = Path(str(summary_meta.get('filtered_csv', '')).strip()) if summary_meta.get('filtered_csv') else None
latest_main = available_summary_filtered_main[0] if available_summary_filtered_main else (available_summary_filtered[0] if available_summary_filtered else None)

use_metadata = False
if _meta_summary:
    if meta_filtered_csv is not None and meta_filtered_csv.exists() and latest_main is not None:
        use_metadata = meta_filtered_csv.stat().st_mtime >= latest_main.stat().st_mtime
    elif latest_main is None:
        use_metadata = True

if use_metadata:
    SUMMARY_STRATEGY = _meta_summary
    SUMMARY_SOURCE = 'metadata'
elif latest_main is not None:
    m_latest = re.search(r'__summary_([a-z0-9_]+?)(?:_uncertainty)?(?:__run_[a-z0-9_]+)?$', latest_main.stem)
    SUMMARY_STRATEGY = m_latest.group(1) if m_latest else DEFAULT_SUMMARY_STRATEGY
    SUMMARY_SOURCE = 'latest_filtered_csv'
else:
    SUMMARY_STRATEGY = DEFAULT_SUMMARY_STRATEGY
    SUMMARY_SOURCE = 'default'

SUMMARY_TOKEN = re.sub(r"[^a-z0-9]+", "_", str(SUMMARY_STRATEGY).strip().lower()).strip("_") or 'province'
DEFAULT_TEMPLATE = PROJECT_DIR / 'inputs/livestock00002.XML'
AVAILABLE_XML_TEMPLATES = sorted((PROJECT_DIR / 'inputs').glob('livestock*.XML'))
DEFAULT_LCI = PROJECT_DIR / f'outputs/CSVs/03-05_espac_livestock_lci_table_filtered_dfe__summary_{SUMMARY_TOKEN}.csv'
if not DEFAULT_LCI.exists():
    DEFAULT_LCI = PROJECT_DIR / f'outputs/CSVs/02_espac_livestock_lci_table_filtered__summary_{SUMMARY_TOKEN}.csv'
DEFAULT_UNCERTAINTY = DEFAULT_LCI.with_name(f"{DEFAULT_LCI.stem}_uncertainty.csv")
XML_EXPORTS_ROOT = PROJECT_DIR / 'outputs/05_xml_exports_livestock_lci'
OUTPUT_DIR = XML_EXPORTS_ROOT / f'summary_{SUMMARY_TOKEN}'

print(f'Inherited livestock summary strategy: {SUMMARY_STRATEGY} (source: {SUMMARY_SOURCE})')
print(f'Expected livestock LCI table: {DEFAULT_LCI}')
print(f'Expected livestock uncertainty table: {DEFAULT_UNCERTAINTY}')
print('Livestock XML templates:')
for template_path in AVAILABLE_XML_TEMPLATES:
    print(' -', template_path.name)

if not AVAILABLE_XML_TEMPLATES:
    display(Markdown('No livestock XML templates found in `inputs/`. Add templates named like `livestock00001.XML`, `livestock00002.XML`, etc., then rerun this notebook.'))

from scripts.pipeline_manifest import new_run_id, make_snapshot_copy, build_manifest_record, append_manifest_record


Inherited livestock summary strategy: region (source: metadata)
Expected livestock LCI table: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_livestock_lci_table_filtered_dfe__summary_region.csv
Expected livestock uncertainty table: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_livestock_lci_table_filtered_dfe__summary_region_uncertainty.csv
Livestock XML templates:
 - livestock00001.XML
 - livestock00002.XML
 - livestock00003.XML


In [8]:

LCI_TABLE_PATH = DEFAULT_LCI
UNCERTAINTY_TABLE_PATH = DEFAULT_UNCERTAINTY

if AVAILABLE_XML_TEMPLATES and LCI_TABLE_PATH.exists():
    df = pd.read_csv(LCI_TABLE_PATH)
    uncertainty_path = UNCERTAINTY_TABLE_PATH
    if uncertainty_path.exists():
        df_unc = pd.read_csv(uncertainty_path)
        unc_cols = [c for c in df_unc.columns if isinstance(c, str) and c.endswith(('__minValue', '__maxValue'))]
        merge_keys = [c for c in ['Region', 'Province', 'Product', 'System'] if c in df.columns and c in df_unc.columns]
        if len(df_unc) == len(df) and unc_cols:
            df = pd.concat([df.reset_index(drop=True), df_unc[unc_cols].reset_index(drop=True)], axis=1)
        elif merge_keys and unc_cols:
            right = df_unc[merge_keys + unc_cols].drop_duplicates(subset=merge_keys, keep='first')
            base = df.reset_index().rename(columns={'index': '__row_id'})
            df = base.merge(right, on=merge_keys, how='left').sort_values('__row_id').drop(columns='__row_id').reset_index(drop=True)
    print(f'Rows: {len(df):,}  Columns: {len(df.columns)}')
    display(df.head(3))


Rows: 4  Columns: 493


,Region,Province,Product,Species,Functional_unit,Normalized_product_output_kg,Area_ha_per_1kg_product,Animals_total_live_weight_kg_per_1kg_product,Producing_animals_live_weight_kg_per_1kg_product,Animals_sold_live_weight_kg_per_1kg_product,...,kgN2O_manure_mgmt_direct_per_1kg_product__minValue,kgN2O_manure_mgmt_direct_per_1kg_product__maxValue,kgN2O_manure_mgmt_indirect_per_1kg_product__minValue,kgN2O_manure_mgmt_indirect_per_1kg_product__maxValue,kgN2O_manure_mgmt_total_per_1kg_product__minValue,kgN2O_manure_mgmt_total_per_1kg_product__maxValue,kgN2_manure_mgmt_per_1kg_product__minValue,kgN2_manure_mgmt_per_1kg_product__maxValue,kgN_manure_after_management_per_1kg_product__minValue,kgN_manure_after_management_per_1kg_product__maxValue
0,(unknown),(all provinces confounded),milk,cattle,1 kg FPCM,1.0,0.284664,9.250598,50.878291,NaN,...,0.000096,0.000138,0.000016,0.000023,0.000112,0.000161,0.000184,0.000264,0.045852,0.065838
1,Costa,(all provinces confounded),milk,cattle,1 kg FPCM,1.0,0.908177,18.501197,101.756581,NaN,...,0.000218,0.000278,0.000036,0.000046,0.000254,0.000324,0.000416,0.000531,0.103744,0.132154
2,Oriente,(all provinces confounded),milk,cattle,1 kg FPCM,1.0,1.098971,18.501197,101.756581,NaN,...,0.000176,0.000252,0.000029,0.000041,0.000205,0.000293,0.000336,0.000481,0.083794,0.119705


In [9]:

def normalize_key(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r'[^a-z0-9_]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s


def to_numeric_or_default(v, default=0.0):
    try:
        if pd.isna(v):
            return float(default)
        return float(v)
    except Exception:
        return float(default)


def safe_name(*parts):
    txt = '_'.join([str(p).strip() for p in parts if str(p).strip() and str(p).strip().lower() != 'nan'])
    txt = re.sub(r'[^A-Za-z0-9._-]+', '_', txt).strip('_')
    return txt[:180] if txt else 'record'


STRATEGY_NUMBER_MAP = {
    'province': 1,
    'region': 2,
    'crop_national': 3,
    'cropping_system': 4,
    'irrig_m3_class': 5,
    'farm_size_class': 6,
    'crop_group': 7,
    'crop_group_national': 8,
    'product': 9,
    'national': 10,
}


def _strategy_code(summary_token: str) -> str:
    token = safe_name(summary_token).lower() or 'unknown'
    number = STRATEGY_NUMBER_MAP.get(token)
    return f'strategy_{number}' if number is not None else f'strategy_{token}'

PASTURE_INPUT_DATASET_NAME = f"forages_pastures | summary: {_strategy_code('crop_group_national')}"

ARTIFICIAL_AREA_PRODUCTS = {'eggs', 'meat_poultry', 'swine_live'}
LAND_USE_NAMES_BY_PRODUCT_GROUP = {
    'artificial_areas': {
        'occupation': 'Occupation, artificial areas, EC',
        'to': 'Transformation, to artificial areas, EC',
        'from': 'Transformation, from unspecified, EC',
    },
    'grassland': {
        'occupation': 'Occupation, grassland/pasture/meadow, EC',
        'to': 'Transformation, to grassland/pasture/meadow, EC',
        'from': 'Transformation, from grassland/pasture/meadow, EC',
    },
}


def _land_use_name_mapping_for_livestock_row(row: pd.Series) -> dict[str, str]:
    product = _clean_display_part(row.get('Product', '')).lower()
    key = 'artificial_areas' if product in ARTIFICIAL_AREA_PRODUCTS else 'grassland'
    return LAND_USE_NAMES_BY_PRODUCT_GROUP[key]


def _clean_display_part(value) -> str:
    raw = str(value or '').strip()
    if not raw or raw.lower() in {'nan', '(unknown)', 'unknown'}:
        return ''
    return raw


def _is_confounded_placeholder(value) -> bool:
    raw_lc = str(value or '').strip().lower()
    return raw_lc.startswith('(all ') or raw_lc.endswith(' confounded)')


def _livestock_display_name_and_slug_parts(row: pd.Series, summary_token: str):
    display_parts = []
    slug_parts = []
    token = normalize_key(summary_token)

    product = _clean_display_part(row.get('Product', ''))
    if product:
        display_parts.append(f'product: {product}')
        slug_parts.append(f"product_{safe_name(product).lower()}")

    system = _clean_display_part(row.get('System', ''))
    include_system = token in {'national', 'province'}
    if system and include_system:
        display_parts.append(f'system: {system}')
        slug_parts.append(f"system_{safe_name(system).lower()}")

    region = _clean_display_part(row.get('Region', ''))
    include_region = token in {'region', 'province'}
    if region and include_region and not _is_confounded_placeholder(region):
        display_parts.append(f'region: {region}')
        slug_parts.append(f"region_{safe_name(region).lower()}")

    province = _clean_display_part(row.get('Province', ''))
    include_province = token == 'province'
    if province and include_province and not _is_confounded_placeholder(province):
        display_parts.append(f'province: {province}')
        slug_parts.append(f"province_{safe_name(province).lower()}")

    strategy_code = _strategy_code(summary_token)
    display_parts.append(f'summary: {strategy_code}')
    slug_parts.append(strategy_code)
    return display_parts, slug_parts


def _livestock_process_name_for_row(row: pd.Series, summary_token: str) -> str:
    display_parts, _ = _livestock_display_name_and_slug_parts(row, summary_token)
    return ' | '.join(display_parts)[:255] if display_parts else 'record'


def _livestock_filename_for_row(row: pd.Series, summary_token: str) -> str:
    _, slug_parts = _livestock_display_name_and_slug_parts(row, summary_token)
    return safe_name(*slug_parts) + '.xml'


def get_root_and_ns(template_xml_path: Path):
    tree = ET.parse(template_xml_path)
    root = tree.getroot()
    m = re.match(r'\{(.+)\}', root.tag)
    ns = {'spold': m.group(1)} if m else {}
    return tree, root, ns


def qname(local_name: str, ns: dict):
    if ns and ns.get('spold'):
        return f"{{{ns['spold']}}}{local_name}"
    return local_name


def parse_template_product_tokens(template_path: Path):
    tree = ET.parse(template_path)
    root = tree.getroot()
    m = re.match(r'\{(.+)\}', root.tag)
    ns = {'spold': m.group(1)} if m else {}
    for ex in root.findall('.//' + qname('exchange', ns)):
        output_group = ex.find(qname('outputGroup', ns))
        if output_group is not None and str(output_group.text or '').strip() == '0':
            comment = str(ex.attrib.get('generalComment', '')).strip()
            if comment:
                return [token.strip().lower() for token in comment.split(',') if token.strip()]
    return []


TEMPLATE_PRODUCT_MAP = {}
for template_path in AVAILABLE_XML_TEMPLATES:
    for token in parse_template_product_tokens(template_path):
        TEMPLATE_PRODUCT_MAP.setdefault(token, template_path)


def template_for_product(product: str):
    if not product:
        return None
    return TEMPLATE_PRODUCT_MAP.get(str(product).strip().lower())


def parse_comment_expression(comment: str):
    parts = [p.strip() for p in re.split(r'\s*\+\s*', str(comment).strip()) if p.strip()]
    return parts


def evaluate_comment_expression(row: pd.Series, comment: str):
    cols = parse_comment_expression(comment)
    if not cols:
        return None
    values = []
    for col in cols:
        if col not in row.index:
            return None
        values.append(to_numeric_or_default(row.get(col), default=0.0))
    return sum(values)


def evaluate_uncertainty(row: pd.Series, comment: str):
    cols = parse_comment_expression(comment)
    if not cols:
        return None
    mins = []
    maxs = []
    for col in cols:
        lo = row.get(f'{col}__minValue')
        hi = row.get(f'{col}__maxValue')
        if pd.isna(lo) or pd.isna(hi):
            return None
        mins.append(to_numeric_or_default(lo, default=0.0))
        maxs.append(to_numeric_or_default(hi, default=0.0))
    return sum(mins), sum(maxs)


LAND_USE_TOTAL_AREA_COLUMN = 'Area_ha_per_1kg_product'
LAND_USE_LINKED_PASTURE_COLUMN = 'Linked_pasture_area_ha_per_1kg_product'


def residual_non_pasture_land_use(row: pd.Series):
    total_area = to_numeric_or_default(row.get(LAND_USE_TOTAL_AREA_COLUMN), default=0.0)
    linked_pasture_area = to_numeric_or_default(row.get(LAND_USE_LINKED_PASTURE_COLUMN), default=0.0)
    residual_area = max(total_area - linked_pasture_area, 0.0)

    total_lo = row.get(f'{LAND_USE_TOTAL_AREA_COLUMN}__minValue')
    total_hi = row.get(f'{LAND_USE_TOTAL_AREA_COLUMN}__maxValue')
    linked_lo = row.get(f'{LAND_USE_LINKED_PASTURE_COLUMN}__minValue')
    linked_hi = row.get(f'{LAND_USE_LINKED_PASTURE_COLUMN}__maxValue')
    if pd.isna(total_lo) or pd.isna(total_hi):
        return residual_area, None

    total_lo_val = to_numeric_or_default(total_lo, default=0.0)
    total_hi_val = to_numeric_or_default(total_hi, default=0.0)
    linked_lo_val = 0.0 if pd.isna(linked_lo) else to_numeric_or_default(linked_lo, default=0.0)
    linked_hi_val = 0.0 if pd.isna(linked_hi) else to_numeric_or_default(linked_hi, default=0.0)

    residual_lo = max(total_lo_val - linked_hi_val, 0.0)
    residual_hi = max(total_hi_val - linked_lo_val, 0.0)
    residual_hi = max(residual_hi, residual_lo)
    return residual_area, (residual_lo, residual_hi)


def _slaughter_species_for_product(product_name: str) -> str | None:
    p = str(product_name or '').strip().lower()
    if not p:
        return None
    if 'poultry' in p or 'chicken' in p or 'egg' in p:
        return 'chicken'
    if 'swine' in p or 'pig' in p:
        return 'swine'
    if 'horse' in p or 'mule' in p or 'donkey' in p:
        return 'equine'
    if 'goat' in p or 'caprine' in p:
        return 'goat'
    if 'ovine' in p or 'sheep' in p or 'lamb' in p:
        return 'sheep'
    if 'milk' in p:
        return 'cattle'
    if 'cattle' in p or 'bovine' in p or 'beef' in p:
        return 'cattle'
    return None


def _slaughter_species_for_exchange(exchange_name: str) -> str | None:
    n = str(exchange_name or '').strip().lower()
    if n.startswith('cattle for slaughtering') or n.startswith('calf,') or n.startswith('dairy cow,'):
        return 'cattle'
    if n.startswith('chicken for slaughtering') or n.startswith('one-day-chicken') or n.startswith('laying hen'):
        return 'chicken'
    if n.startswith('swine for slaughtering') or n.startswith('piglet,'):
        return 'swine'
    if n.startswith('kid goat'):
        return 'goat'
    if n.startswith('sheep for slaughtering') or n.startswith('lamb,'):
        return 'sheep'
    return None


def _feed_exchange_species(exchange_name: str) -> set[str]:
    n = str(exchange_name or '').strip().lower()
    species = set()
    if 'beef cattle' in n or 'dairy cow' in n:
        species.add('cattle')
    if 'swine' in n or 'pig' in n:
        species.add('swine')
    if 'broiler' in n or 'laying hen' in n or 'chicken' in n or 'poultry' in n:
        species.add('chicken')
    if 'forages_pastures' in n or 'pasture' in n:
        species.update({'cattle', 'sheep', 'goat', 'equine'})
    return species


def _animal_input_column_for_exchange(exchange_name: str) -> str | None:
    n = str(exchange_name or '').strip().lower()
    if n.startswith('calf,'):
        return 'Animal_input_calf_live_weight_kg_per_1kg_product'
    if n.startswith('one-day-chicken'):
        return 'Animal_input_one_day_chicken_live_weight_kg_per_1kg_product'
    if n.startswith('piglet,'):
        return 'Animal_input_piglet_live_weight_kg_per_1kg_product'
    if n.startswith('kid goat'):
        return 'Animal_input_kid_goat_live_weight_kg_per_1kg_product'
    if n.startswith('dairy cow,'):
        return 'Animal_input_dairy_cow_live_weight_kg_per_1kg_product'
    if n.startswith('laying hen'):
        return 'Animal_input_laying_hen_live_weight_kg_per_1kg_product'
    return None


def apply_row_to_xml(tree, root, ns, row: pd.Series):
    for ex in root.findall('.//' + qname('exchange', ns)):
        comment = str(ex.attrib.get('generalComment', '')).strip()
        if not comment:
            continue
        # Set exchange name for output exchange (outputGroup == '0')
        output_group = ex.find(qname('outputGroup', ns))
        if output_group is not None and str(output_group.text or '').strip() == '0':
            product_name = str(row.get('Product', '')).strip()
            system_name = str(row.get('System', '')).strip()
            if product_name:
                ex.attrib['name'] = f"{product_name}, {system_name}".rstrip(', ')
        value = evaluate_comment_expression(row, comment)
        if value is None:
            continue
        ex.attrib['meanValue'] = str(value)
        uncertainty = evaluate_uncertainty(row, comment)
        if uncertainty is not None:
            lo, hi = uncertainty
            ex.attrib['uncertaintyType'] = '3'
            ex.attrib['minValue'] = str(lo)
            ex.attrib['maxValue'] = str(hi)

    for aname in root.findall('.//' + qname('activityName', ns)):
        aname.text = safe_name(row.get('Product', ''), row.get('System', ''), row.get('Region', ''), row.get('Province', ''))
    return tree



In [10]:
# SimaPro XML Correction Constants and Functions
DEFAULT_XML_REVIEWER_NAME = "ESPAC Project"
DEFAULT_XML_VALIDATION_METADATA = {
    "reviewStatus": "draft",
    "reviewer": DEFAULT_XML_REVIEWER_NAME,
    "proofReadingValidator": {
        "name": DEFAULT_XML_REVIEWER_NAME,
        "email": "n/a",
        "companyCode": "ESPAC"
    }
}

SIMAPRO_SCHEMA_LOCATION = (
    "http://www.EcoInvent.org/EcoSpold01 "
    "http://www.EcoInvent.org/EcoSpold01..\\..\\EcoSpold01Dataset.xsd "
    "https://github.com/brightway-lca/pyecospold/tree/main/pyecospold/schemas/v1"
)

SIMAPRO_CLASSIFICATION = {
    "category": "Agricultural",
    "localCategory": "Agricultural",
    "subCategory": "ECUADOR",
    "localSubCategory": "ECUADOR"
}

REFERENCE_PRODUCT_LOCATION = "EC"

XML_VALIDATION_METADATA = DEFAULT_XML_VALIDATION_METADATA.copy()


def _qname(local_name: str, ns: dict) -> str:
    if ns and ns.get("spold"):
        return f"{{{ns['spold']}}}{local_name}"
    return local_name


def _find_or_create(parent, child_name: str, ns: dict):
    child = parent.find(f"spold:{child_name}", ns) if ns else parent.find(child_name)
    if child is None:
        child = ET.SubElement(parent, _qname(child_name, ns))
    return child


def _reorder_modelling_and_validation_children(modelling, representativeness, validation, source):
    children = list(modelling)
    ordered = []
    for node in (representativeness, validation, source):
        if node is not None and node in children and node not in ordered:
            ordered.append(node)
    for node in children:
        if node not in ordered:
            ordered.append(node)
    for node in list(modelling):
        modelling.remove(node)
    for node in ordered:
        modelling.append(node)


def _prepare_tree_for_simapro_serialization(tree, ns: dict):
    if ns and ns.get("spold"):
        ET.register_namespace("", ns["spold"])
    ET.register_namespace("xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root = tree.getroot()
    root.attrib["{http://www.w3.org/2001/XMLSchema-instance}schemaLocation"] = SIMAPRO_SCHEMA_LOCATION


def _ensure_simapro_validation_metadata(root, ns: dict, validation_meta: dict, generation_date: str):
    meta = root.find('.//spold:metaInformation', ns) if ns else root.find('.//metaInformation')
    if meta is None:
        return
    modelling = _find_or_create(meta, 'modellingAndValidation', ns)
    representativeness = _find_or_create(modelling, 'representativeness', ns)
    source = _find_or_create(modelling, 'source', ns)
    validation = _find_or_create(modelling, 'validation', ns)

    def _set_text(parent, child_name, value):
        child = _find_or_create(parent, child_name, ns)
        child.text = "" if value is None else str(value)

    _set_text(validation, 'reviewStatus', validation_meta.get('reviewStatus', 'draft'))
    _set_text(validation, 'reviewer', validation_meta.get('reviewer', ''))
    _set_text(validation, 'reviewDate', generation_date)

    validator_meta = validation_meta.get('proofReadingValidator', {}) or {}
    proof = _find_or_create(validation, 'proofReadingValidator', ns)
    _set_text(proof, 'name', validator_meta.get('name', ''))
    _set_text(proof, 'email', validator_meta.get('email', ''))
    _set_text(proof, 'companyCode', validator_meta.get('companyCode', ''))
    _set_text(proof, 'proofReadingDate', generation_date)

    _reorder_modelling_and_validation_children(modelling, representativeness, validation, source)


def _apply_exchange_location_unit_rules(ex, geography_location: str):
    # SimaPro requires exchange #1 location to match dataset geography.
    number_raw = str(ex.attrib.get("number", "")).strip()
    try:
        number = int(number_raw)
    except Exception:
        number = None

    if number == 1:
        ex.attrib["location"] = REFERENCE_PRODUCT_LOCATION
    else:
        ex.attrib["location"] = "RER"

    if number is not None and 19 <= number <= 25:
        ex.attrib["unit"] = "kg"


def _ensure_exchange_group_tag(ex, ns: dict):
    def _lname(tag):
        return tag.split("}", 1)[1] if isinstance(tag, str) and tag.startswith("{") and "}" in tag else str(tag)

    has_group = any(_lname(child.tag) in {"inputGroup", "outputGroup"} for child in list(ex))
    if has_group:
        return

    # Elementary/resource flows use outputGroup=4; technosphere defaults to inputGroup=5.
    if "category" in ex.attrib:
        group = ET.SubElement(ex, _qname("outputGroup", ns))
        group.text = "4"
    else:
        group = ET.SubElement(ex, _qname("inputGroup", ns))
        group.text = "5"


def get_exchange_elements(root, ns: dict):
    if ns:
        return root.findall('.//spold:flowData/spold:exchange', ns)
    return root.findall('.//flowData/exchange')

In [11]:
def _validate_and_correct_uncertainty_bounds(mean_val: float, min_val: float, max_val: float) -> tuple:
    """
    Ensure that minValue <= meanValue <= maxValue.
    If bounds violate the constraint, expand them to encompass the mean value.
    
    Returns: (corrected_min, corrected_max)
    """
    # Expand lower bound if needed
    if min_val > mean_val:
        min_val = mean_val
    # Expand upper bound if needed
    if max_val < mean_val:
        max_val = mean_val
    # Ensure min <= max (though they should be after above operations)
    if min_val > max_val:
        min_val, max_val = max_val, min_val
    return float(min_val), float(max_val)


def render_xml_for_livestock_row(template_xml_path: Path, row: pd.Series, generation_date=None):
    """
    Render corrected XML for a livestock row with SimaPro compatibility.
    Applies namespace registration, validation metadata, location/unit rules, and exchange groups.
    """
    tree, root, ns = get_root_and_ns(template_xml_path)
    generation_date = generation_date or date.today().isoformat()
    
    # Apply SimaPro validation metadata
    _ensure_simapro_validation_metadata(root, ns, XML_VALIDATION_METADATA, generation_date)
    
    # Get geography location for reference exchange
    if ns:
        geo = root.find('.//spold:geography', ns)
    else:
        geo = root.find('.//geography')
    if geo is not None:
        geo.attrib["location"] = REFERENCE_PRODUCT_LOCATION
    geography_location = REFERENCE_PRODUCT_LOCATION
    
    product_name_for_row = str(row.get('Product', '')).strip()
    target_slaughter_species = _slaughter_species_for_product(product_name_for_row)

    # Apply corrections to each exchange
    for ex in get_exchange_elements(root, ns):
        # Apply location and unit rules
        _apply_exchange_location_unit_rules(ex, geography_location)
        # Ensure inputGroup/outputGroup tags exist
        _ensure_exchange_group_tag(ex, ns)

        ex_name_raw = str(ex.attrib.get('name', '')).strip()
        ex_name = ex_name_raw.lower()
        land_use_mapping = _land_use_name_mapping_for_livestock_row(row)
        if ex_name.startswith('occupation, '):
            ex.attrib['name'] = land_use_mapping['occupation']
            ex_name_raw = ex.attrib['name']
            ex_name = ex_name_raw.lower()
        elif ex_name.startswith('transformation, to '):
            ex.attrib['name'] = land_use_mapping['to']
            ex_name_raw = ex.attrib['name']
            ex_name = ex_name_raw.lower()
        elif ex_name.startswith('transformation, from '):
            ex.attrib['name'] = land_use_mapping['from']
            ex_name_raw = ex.attrib['name']
            ex_name = ex_name_raw.lower()

        if ex_name == 'forages_pastures':
            ex.attrib['name'] = PASTURE_INPUT_DATASET_NAME

        is_land_use_exchange = (
            ex_name.startswith('occupation, ')
            or ex_name.startswith('transformation, to ')
            or ex_name.startswith('transformation, from ')
        )

        comment = str(ex.attrib.get("generalComment", "")).strip()
        if not comment:
            continue
        
        # Set exchange name for output exchange (outputGroup == '0')
        output_group = ex.find(_qname('outputGroup', ns))
        if output_group is not None and str(output_group.text or "").strip() == "0":
            product_name = str(row.get('Product', '')).strip()
            system_name = str(row.get('System', '')).strip()
            if product_name:
                ex.attrib['name'] = f"{product_name}, {system_name}".rstrip(', ')
        
        # Livestock water is represented by Water_l_per_1kg_product in the livestock system,
        # not by crop irrigation source exchanges.
        if comment == "Irrig_m3":
            continue

        # Evaluate exchange value from row
        if is_land_use_exchange:
            value, uncertainty = residual_non_pasture_land_use(row)
        else:
            value = evaluate_comment_expression(row, comment)
            uncertainty = evaluate_uncertainty(row, comment)
        routed_animal_column = None
        if comment == 'Animals_total_live_weight_kg_per_1kg_product':
            routed_animal_column = _animal_input_column_for_exchange(ex_name_raw)
            if routed_animal_column and routed_animal_column in row.index:
                value = evaluate_comment_expression(row, routed_animal_column)
                uncertainty = evaluate_uncertainty(row, routed_animal_column)

        # Route feed exchanges by species so each inventory only carries its own feed profile.
        if 'feed' in comment.lower():
            feed_species = _feed_exchange_species(ex_name_raw)
            if feed_species and (target_slaughter_species is None or target_slaughter_species not in feed_species):
                value = 0.0
                uncertainty = (0.0, 0.0)

        # Shared livestock live-weight comment appears in multiple animal exchanges.
        # Keep non-zero only for the animal matching the current product.
        if (
            comment == 'Animals_total_live_weight_kg_per_1kg_product'
            and target_slaughter_species is not None
            and routed_animal_column is None
        ):
            exchange_species = _slaughter_species_for_exchange(ex_name_raw)
            if exchange_species is not None and exchange_species != target_slaughter_species:
                value = 0.0
                uncertainty = (0.0, 0.0)
        if value is not None:
            ex.attrib['meanValue'] = str(value)
        
        # Evaluate and set uncertainty bounds
        if uncertainty is not None:
            lo, hi = uncertainty
            ex.attrib['uncertaintyType'] = '3'
            # Bounds already correct from aggregation - no validation needed
            mean_val = float(ex.attrib.get('meanValue', 0))
            # lo, hi already set from resolve_*_uncertainty_bounds functions
            ex.attrib['minValue'] = str(lo)
            ex.attrib['maxValue'] = str(hi)
            ex.attrib['maxValue'] = str(hi)
        else:
            # For non-reference exchanges, still set uncertainty type
            if value is not None:
                number_raw = str(ex.attrib.get("number", "")).strip()
                try:
                    number = int(number_raw)
                    if number != 1:  # Don't apply uncertainty to reference exchange
                        ex.attrib["uncertaintyType"] = "3"
                        ex.attrib["minValue"] = str(value)
                        ex.attrib["maxValue"] = str(value)
                except Exception:
                    pass
    
    # Set reference product and process names
    if ns:
        ref = root.find('.//spold:referenceFunction', ns)
        ex1 = root.find('.//spold:flowData/spold:exchange[@number="1"]', ns)
    else:
        ref = root.find('.//referenceFunction')
        ex1 = root.find('.//flowData/exchange[@number="1"]')
    
    proc_name = _livestock_process_name_for_row(row, SUMMARY_TOKEN)
    
    if ref is not None:
        ref.attrib['name'] = proc_name
        ref.attrib['localName'] = proc_name
        ref.attrib['category'] = SIMAPRO_CLASSIFICATION.get("category", "Agricultural")
        ref.attrib['localCategory'] = SIMAPRO_CLASSIFICATION.get("localCategory", "Agricultural")
        ref.attrib['subCategory'] = SIMAPRO_CLASSIFICATION.get("subCategory", "ECUADOR")
        ref.attrib['localSubCategory'] = SIMAPRO_CLASSIFICATION.get("localSubCategory", "ECUADOR")
    
    if ex1 is not None:
        ex1.attrib['name'] = proc_name
        # SimaPro compatibility: keep reference product exchange free of uncertainty bounds
        for attr in ('uncertaintyType', 'minValue', 'maxValue'):
            ex1.attrib.pop(attr, None)
    
    # Update activity name
    for aname in root.findall('.//' + _qname('activityName', ns)):
        aname.text = proc_name
    
    # Prepare tree for serialization (namespace registration and schemaLocation)
    _prepare_tree_for_simapro_serialization(tree, ns)
    
    return tree


In [12]:

written = []
if AVAILABLE_XML_TEMPLATES and LCI_TABLE_PATH.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    generation_date = date.today().isoformat()
    for idx, row in df.iterrows():
        product = str(row.get('Product', '')).strip()
        if not product or '(unknown)' in product.lower():
            continue  # Skip rows with empty or unknown product
        # Skip rows with too many empty columns (malformed data)
        non_empty_count = sum(1 for v in row.values if pd.notna(v) and str(v).strip())
        if non_empty_count < 20:  # Adjust threshold as needed
            continue
        system = str(row.get('System', '')).strip().replace('(unknown)', '').strip()
        region = str(row.get('Region', '')).strip().replace('(unknown)', '').strip()
        province = str(row.get('Province', '')).strip().replace('(unknown)', '').strip()
        template_path = template_for_product(product)
        if template_path is None:
            if DEFAULT_TEMPLATE.exists():
                template_path = DEFAULT_TEMPLATE
            else:
                template_path = AVAILABLE_XML_TEMPLATES[0]
        # Use corrected rendering function with SimaPro compatibility
        tree = render_xml_for_livestock_row(template_path, row, generation_date=generation_date)
        filename = _livestock_filename_for_row(row, SUMMARY_TOKEN)
        out_path = OUTPUT_DIR / filename
        tree.write(out_path, encoding='UTF-8', xml_declaration=True)
        written.append(out_path)
    print(f'Wrote {len(written):,} livestock XML files to {OUTPUT_DIR}')
    if written:
        display(Markdown(f'Sample output: `{written[0]}`'))

if written:
    livestock_xml_run_id = new_run_id('05_xml_livestock')
    main_snapshot = make_snapshot_copy(LCI_TABLE_PATH, livestock_xml_run_id)
    unc_path = UNCERTAINTY_TABLE_PATH if UNCERTAINTY_TABLE_PATH.exists() else LCI_TABLE_PATH.with_name(f"{LCI_TABLE_PATH.stem}_uncertainty.csv")
    unc_snapshot = make_snapshot_copy(unc_path, livestock_xml_run_id) if unc_path.exists() else main_snapshot
    rec = build_manifest_record(
        run_id=livestock_xml_run_id,
        domain='livestock',
        summary_token=SUMMARY_TOKEN,
        pipeline_stage='05_xml',
        source_main_csv=main_snapshot,
        source_unc_csv=unc_snapshot,
        main_df=df,
        unc_df=df[[c for c in df.columns if c.endswith('__minValue') or c.endswith('__maxValue')]].copy() if isinstance(df, pd.DataFrame) else pd.DataFrame(),
        filters_meta=summary_meta if isinstance(summary_meta, dict) else {},
        upstream_run_id=str((summary_meta or {}).get('run_id', '') or None),
        extra={'xml_output_dir': str(OUTPUT_DIR.resolve()), 'xml_files_written': int(len(written)), 'otros_expansion_source': 'none', 'subcrop_count_by_label': {}},
    )
    mpath = append_manifest_record(PROJECT_DIR, rec)
    print(f"Manifest updated: {mpath} (run_id={livestock_xml_run_id})")


Wrote 4 livestock XML files to c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\05_xml_exports_livestock_lci\summary_region


Sample output: `c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\05_xml_exports_livestock_lci\summary_region\product_milk_strategy_2.xml`

Manifest updated: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\pipeline_run_manifest.json (run_id=05_xml_livestock_20260506T103739Z)
